# Predicting Nutrient Gaps


In [28]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, VotingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import pickle
import joblib

import os

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load datasets

path = '../dataset/'
train_path = os.path.join(path, 'Train.csv')
test_path = os.path.join(path, 'Test.csv')
gap_train_path = os.path.join(path, 'Gap_Train.csv')
gap_test_path = os.path.join(path, 'Gap_Test.csv')
sample_submission_path = os.path.join(path, 'SampleSubmission.csv')
# Load the datasets 
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_gap_df = pd.read_csv(gap_train_path)
test_gap_df = pd.read_csv(gap_test_path)
sample_submission = pd.read_csv(sample_submission_path)

In [3]:
train_df.head()

,site,PID,lon,lat,pH,alb,bio1,bio12,bio15,bio7,...,P,K,Ca,Mg,S,Fe,Mn,Zn,Cu,B
0,site_id_bIEHwl,ID_I5RGjv,70.603761,46.173798,7.75,176,248,920,108,190,...,0.34,147,6830,2310,5.66,75.2,85.0,0.82,2.98,0.24
1,site_id_nGvnKc,ID_8jWzJ5,70.590479,46.078924,7.10,181,250,1080,113,191,...,11.70,151,1180,235,19.40,96.2,409.0,2.57,4.32,0.10
2,site_id_nGvnKc,ID_UgzkN8,70.582553,46.048820,6.95,188,250,1109,111,191,...,21.80,151,1890,344,11.00,76.7,65.0,1.95,1.24,0.22
3,site_id_nGvnKc,ID_DLLHM9,70.573267,46.021910,7.83,174,250,1149,112,191,...,39.90,201,6660,719,14.90,81.9,73.0,4.90,3.08,0.87
4,site_id_7SA9rO,ID_d009mj,70.585330,46.204336,8.07,188,250,869,114,191,...,1.00,90,7340,1160,8.66,69.4,149.0,0.55,3.03,0.31


In [4]:
test_df.head()

,site,PID,lon,lat,pH,alb,bio1,bio12,bio15,bio7,...,para,parv,ph20,slope,snd20,soc20,tim,wp,xhp20,BulkDensity
0,site_id_hgJpkz,ID_NGS9Bx,69.170794,44.522885,6.86,144,256,910,108,186,...,37.940418,467.619293,6.825,1.056416,25.50,15.25,8.732471,0.016981,0.005831,1.20
1,site_id_olmuI5,ID_YdVKXw,68.885265,44.741057,7.08,129,260,851,110,187,...,35.961353,542.590149,6.725,0.730379,18.75,14.00,10.565657,0.021030,0.005134,1.24
2,site_id_PTZdJz,ID_MZAlfE,68.970210,44.675777,6.50,142,259,901,109,187,...,38.983898,416.385437,6.825,1.146542,21.00,14.00,9.590125,0.018507,0.004480,1.23
3,site_id_DOTgr8,ID_GwCCMN,69.068751,44.647707,6.82,142,261,847,109,187,...,39.948471,374.971008,6.725,0.567210,23.25,12.25,9.669279,0.021688,0.006803,1.22
4,site_id_1rQNvy,ID_K8sowf,68.990002,44.577607,6.52,145,253,1109,110,186,...,33.658615,361.233643,6.200,1.169207,26.25,18.25,7.895920,0.023016,0.000874,1.23


In [5]:
train_gap_df.head()

,Nutrient,Required,Available,Gap,PID
0,N,100.0,3796.0000,-3696.0000,ID_I5RGjv
1,P,40.0,0.9928,39.0072,ID_I5RGjv
2,K,52.0,429.2400,-377.2400,ID_I5RGjv
3,Ca,12.0,19943.6000,-19931.6000,ID_I5RGjv
4,Mg,8.0,6745.2000,-6737.2000,ID_I5RGjv


In [6]:
test_gap_df = pd.merge(test_gap_df, test_df[['PID', 'BulkDensity']], on='PID', how='left')

In [7]:
test_gap_df.head()

,Nutrient,Required,PID,BulkDensity
0,N,100.0,ID_NGS9Bx,1.2
1,P,40.0,ID_NGS9Bx,1.2
2,K,52.0,ID_NGS9Bx,1.2
3,Ca,12.0,ID_NGS9Bx,1.2
4,Mg,8.0,ID_NGS9Bx,1.2


In [8]:
sample_submission.head()

,ID,Gap
0,ID_002W8m_B,0
1,ID_002W8m_Ca,0
2,ID_002W8m_Cu,0
3,ID_002W8m_Fe,0
4,ID_002W8m_K,0


In [9]:
# Display basic info
print("Train Data Info:")
print(train_df.info())
print("\nTest Data Info:")
print(test_df.info())

Train Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7744 entries, 0 to 7743
Data columns (total 44 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   site         7744 non-null   object 
 1   PID          7744 non-null   object 
 2   lon          7744 non-null   float64
 3   lat          7744 non-null   float64
 4   pH           7744 non-null   float64
 5   alb          7744 non-null   int64  
 6   bio1         7744 non-null   int64  
 7   bio12        7744 non-null   int64  
 8   bio15        7744 non-null   int64  
 9   bio7         7744 non-null   int64  
 10  bp           7744 non-null   float64
 11  cec20        7744 non-null   float64
 12  dows         7744 non-null   float64
 13  ecec20       7739 non-null   float64
 14  hp20         7739 non-null   float64
 15  ls           7744 non-null   float64
 16  lstd         7744 non-null   float64
 17  lstn         7744 non-null   float64
 18  mb1          7744 non-null   fl

In [10]:
# prompt: input missing values in train_df and test_df with the mean, only do it for columns that have missing values

# Fill missing values with the mean for columns with missing values in train_df
for column in train_df.columns:
  if train_df[column].isnull().any():
    train_df[column].fillna(train_df[column].mean(), inplace=True)

# Fill missing values with the mean for columns with missing values in test_df
for column in test_df.columns:
  if test_df[column].isnull().any():
    test_df[column].fillna(test_df[column].mean(), inplace=True)


# Step 6: Let's model ⏳

In [11]:
target_columns = ['N', 'P', 'K', 'Ca', 'Mg', 'S', 'Fe', 'Mn', 'Zn', 'Cu', 'B']

In [12]:
# Feature selection
X = train_df.drop(columns=target_columns)
y = train_df[target_columns]
X_test = test_df.drop(columns=['PID',"site"])

In [13]:
# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
X_train = X_train.drop(columns=['PID','site'])
X_val = X_val.drop(columns=['PID','site'])

In [15]:
model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))
model.fit(X_train, y_train)

,estimator,RandomForestR...ndom_state=42)
,n_jobs,None
,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0


In [16]:
# Predict on validation set
predictions = model.predict(X_test)
y_pred = model.predict(X_val)

In [17]:
# Evaluate model
mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
print(f'MAE: {mae:.4f}, RMSE: {rmse:.4f}')

MAE: 159.8839, RMSE: 480.8662


In [18]:
test_predictions = model.predict(X_test)

In [19]:
# Split the predictions into separate columns
N_pred =  test_predictions[:, 0]  # Predictions for N
P_pred =  test_predictions[:, 1]  # Predictions for P
K_pred =  test_predictions[:, 2]  # Predictions for K
Ca_pred = test_predictions[:, 3]  # Predictions for Ca
Mg_pred = test_predictions[:, 4]  # Predictions for Mg
S_pred =  test_predictions[:, 5]  # Predictions for S
Fe_pred = test_predictions[:, 6]  # Predictions for Fe
Mn_pred = test_predictions[:, 7]  # Predictions for Mn
Zn_pred = test_predictions[:, 8]  # Predictions for Zn
Cu_pred = test_predictions[:, 9]  # Predictions for Cu
B_pred =  test_predictions[:, 10]  # Predictions for B


In [20]:
submission = pd.DataFrame({'PID': test_df['PID'], 'N': N_pred, 'P': P_pred, 'K': K_pred, 'Ca': Ca_pred, 'Mg': Mg_pred, 'S': S_pred, 'Fe': Fe_pred, 'Mn': Mn_pred, 'Zn': Zn_pred, 'Cu': Cu_pred, 'B': B_pred})
submission.head()

,PID,N,P,K,Ca,Mg,S,Fe,Mn,Zn,Cu,B
0,ID_NGS9Bx,1737.4,15.4269,179.27,5956.97,1673.80,10.5150,129.765,147.887,1.8131,4.5936,0.2113
1,ID_YdVKXw,1385.0,8.9349,178.03,6395.83,2328.98,10.0201,119.243,130.930,1.5662,4.5469,0.2065
2,ID_MZAlfE,1925.5,3.9915,186.44,5452.06,1723.34,8.8659,132.232,161.940,1.6959,4.4342,0.2356
3,ID_GwCCMN,1869.5,1.7059,172.56,5737.37,1886.07,9.7395,136.026,162.780,1.8275,4.1322,0.2368
4,ID_K8sowf,1668.3,7.4646,206.42,5928.65,1336.39,8.4816,122.090,128.877,2.0823,4.6589,0.2035


In [21]:
# prompt: turn submission into a 3 column file that has the column PID, Nutrient, Value

submission_melted = submission.melt(id_vars=['PID'], var_name='Nutrient', value_name='Available_Nutrients_in_ppm')
submission_melted = submission_melted.sort_values('PID')
submission_melted.head()

,PID,Nutrient,Available_Nutrients_in_ppm
19869,ID_002W8m,Zn,3.3446
15033,ID_002W8m,Fe,192.1800
2943,ID_002W8m,P,8.0329
24705,ID_002W8m,B,0.4422
525,ID_002W8m,N,2473.7000


In [22]:
# prompt: merge test_gap_df with submission_melted on PID and Nutrient
nutrient_df = pd.merge(test_gap_df, submission_melted, on=['PID', 'Nutrient'], how='left')


In [23]:
soil_depth = 20  # cm

# Calculate the Available_Nutrients_in_kg_ha
nutrient_df['Available_Nutrients_in_kg_ha'] = (nutrient_df['Available_Nutrients_in_ppm']
                                               * soil_depth * nutrient_df['BulkDensity'] * 0.1)

In [24]:
nutrient_df.head()

,Nutrient,Required,PID,BulkDensity,Available_Nutrients_in_ppm,Available_Nutrients_in_kg_ha
0,N,100.0,ID_NGS9Bx,1.2,1737.4000,4169.76000
1,P,40.0,ID_NGS9Bx,1.2,15.4269,37.02456
2,K,52.0,ID_NGS9Bx,1.2,179.2700,430.24800
3,Ca,12.0,ID_NGS9Bx,1.2,5956.9700,14296.72800
4,Mg,8.0,ID_NGS9Bx,1.2,1673.8000,4017.12000


In [25]:
nutrient_df["Gap"] = nutrient_df["Required"] - nutrient_df["Available_Nutrients_in_kg_ha"]

In [26]:
nutrient_df['ID'] = nutrient_df['PID'] + "_" + nutrient_df['Nutrient']
nutrient_df = nutrient_df[['ID', 'Gap']]
nutrient_df.head()

,ID,Gap
0,ID_NGS9Bx_N,-4069.76000
1,ID_NGS9Bx_P,2.97544
2,ID_NGS9Bx_K,-378.24800
3,ID_NGS9Bx_Ca,-14284.72800
4,ID_NGS9Bx_Mg,-4009.12000


If a value is negative it means there is excess of that nutrient in the soil already and the farmer does not need to add any more. If the value is positive then the farmer needs to add those nutrients to the soil.

In [27]:
nutrient_df.to_csv('submission.csv', index=False)
print("Submission file saved as submission.csv")

Submission file saved as submission.csv


In [ ]:
def run_models(X_train, y_train, X_val, y_val):
    """
    Trains, evaluates, and saves multiple regression models, then creates a voting ensemble.

    Args:
        X_train (pd.DataFrame): Training feature data.
        y_train (pd.DataFrame): Training target data.
        X_val (pd.DataFrame): Validation feature data.
        y_val (pd.DataFrame): Validation target data.

    Returns:
        MultiOutputRegressor: The trained multi-output voting regressor model.
    """
    # Define the base models with specified parameters
    base_models = {
        'lightgbm': LGBMRegressor(n_estimators=500, random_state=42),
        'catboost': CatBoostRegressor(n_estimators=500, random_state=42, verbose=0),
        'adaboost': AdaBoostRegressor(n_estimators=500, random_state=42),
        'decision_tree': DecisionTreeRegressor(random_state=42),
        'random_forest': RandomForestRegressor(n_estimators=500, random_state=42),

        'pls_regression': PLSRegression(n_components=min(X_train.shape[1], 10)),
        'linear_regression': LinearRegression()
    }

    rmse_scores = {}
    trained_models = []

    print("--- Training Individual Models ---")
    # Loop through each model, train, evaluate, and save
    for name, model in base_models.items():
        print(f"Training {name}...")
        # Wrap the base model in MultiOutputRegressor
        multi_output_model = MultiOutputRegressor(model)

        # Train the model
        multi_output_model.fit(X_train, y_train)

        # Predict on the validation set
        y_pred = multi_output_model.predict(X_val)

        # Calculate and store RMSE
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_scores[name] = rmse
        print(f"  RMSE for {name}: {rmse:.4f}")

        # Save the trained model to a pickle file
        with open(f'{name}_model.pkl', 'wb') as f:
            pickle.dump(multi_output_model, f)
        print(f"  Model saved as {name}_model.pkl")

        # Add the base model to the list for the voting regressor
        trained_models.append((name, model))

    # Determine the best model based on RMSE
    best_model_name = min(rmse_scores, key=rmse_scores.get)
    print(f"\n--- Best Performing Model: {best_model_name} (RMSE: {rmse_scores[best_model_name]:.4f}) ---")

    # --- Create and Train Voting Regressor ---
    print("\n--- Training Voting Regressor ---")
    voting_regressor = VotingRegressor(estimators=trained_models)

    # Wrap the voting regressor for multi-output prediction
    multi_output_voting_model = MultiOutputRegressor(voting_regressor)

    # Train the voting model
    multi_output_voting_model.fit(X_train, y_train)

    # Evaluate the voting model
    y_pred_voting = multi_output_voting_model.predict(X_val)
    rmse_voting = np.sqrt(mean_squared_error(y_val, y_pred_voting))
    print(f"  RMSE for Voting Regressor: {rmse_voting:.4f}")

    # Save the final voting model
    with open('voting_model.pkl', 'wb') as f:
        pickle.dump(multi_output_voting_model, f)
    print("  Voting model saved as voting_model.pkl")

    return multi_output_voting_model

# Example usage with your dataframes from the notebook:
#
# Assuming X_train, y_train, X_val, y_val are already defined
# final_voting_model = run_models(X_train, y_train, X_val, y_val)
#
# To make predictions on the test set:
# test_predictions = final_voting_model.predict(X_test)
